# RandBox trên Kaggle cho IP102\n
\n
Notebook này tạo split continual learning cho IP102, mặc định chạy thử Task 1 trên 2 GPU của Kaggle, rồi test checkpoint của task đó.\n
\n
Thiết lập được dùng ở đây:\n
- Task 1: 27 lớp\n
- Task 2: 25 lớp\n
- Task 3: 25 lớp\n
- Task 4: 25 lớp\n
\n
Notebook này tự clone repo RandBox, tự cài dependency cơ bản và dùng checkpoint mặc định của Detectron2 nếu bạn không chỉ định checkpoint riêng.\n
Nếu muốn dùng checkpoint pretrain từ README, chỉ cần đổi `PRETRAIN_WEIGHTS` ở cell cấu hình.

In [ ]:
from pathlib import Path
import json
import math
import os
import subprocess
import sys

def run_shell(command):
    print('>>>', command)
    subprocess.run(command, shell=True, check=True)

run_shell(f'{sys.executable} -m pip install -U pip setuptools wheel')
run_shell(f'{sys.executable} -m pip install pycocotools opencv-python-headless matplotlib fvcore iopath omegaconf')

REPO_DIR = Path('/kaggle/working/RandBox')
if not (REPO_DIR / 'train_net.py').exists():
    if not Path('/kaggle/working/.git').exists():
        run_shell('git clone https://github.com/scuwyh2000/RandBox.git /kaggle/working/RandBox')
    else:
        print('RandBox repo already present.')

# Detectron2 install is the only heavyweight dependency. If Kaggle has internet enabled, this will install it automatically.
try:
    import detectron2
    print('Detectron2 already installed.')
except Exception:
    run_shell(f'{sys.executable} -m pip install git+https://github.com/facebookresearch/detectron2.git')

os.chdir(REPO_DIR)
print('Repository:', REPO_DIR)
print('Python:', sys.version)

try:
    import torch
    print('CUDA available:', torch.cuda.is_available())
    print('GPU count:', torch.cuda.device_count())
    if torch.cuda.is_available():
        for idx in range(torch.cuda.device_count()):
            print(idx, torch.cuda.get_device_name(idx))
except Exception as exc:
    print('Torch check failed:', exc)

DATASET_ANN = Path('/kaggle/input/datasets/eljazouly/ip102-coco-annotations/coco_annotations')
IMAGE_ROOT = Path('/kaggle/input/datasets/rtlmhjbn/ip02-dataset/classification/train')
WORK_DATASETS = REPO_DIR / 'datasets'
OUTPUT_ROOT = Path('/kaggle/working/randbox_outputs')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Default to the ImageNet pretrained weight that Detectron2 knows how to resolve.
# Replace this with a Kaggle input path if you uploaded a specific RandBox checkpoint from the README.
PRETRAIN_WEIGHTS = 'detectron2://ImageNetPretrained/torchvision/R-50.pkl'

# Kaggle 2-GPU run. If Kaggle only gives 1 GPU, change NUM_GPUS to 1.
NUM_GPUS = 2

# 2-GPU batch size. If you hit OOM, lower this to 6 or 4. If you have headroom, try 10 or 12.
IMS_PER_BATCH = 8

In [ ]:
TASKS = {
    't1': {'start': 0, 'end': 27, 'prev': 0, 'curr': 27, 'config': 'configs/t1.yaml'},
    't2_ft': {'start': 27, 'end': 52, 'prev': 27, 'curr': 25, 'config': 'configs/t2_ft.yaml'},
    't3_ft': {'start': 52, 'end': 77, 'prev': 52, 'curr': 25, 'config': 'configs/t3_ft.yaml'},
    't4_ft': {'start': 77, 'end': 102, 'prev': 77, 'curr': 25, 'config': 'configs/t4_ft.yaml'},
}

# Chạy thử 1 task trước. Đổi thành list(TASKS) khi muốn train/eval cả 4 task.
ACTIVE_TASKS = ['t1']

def load_coco(path):
    with open(path, 'r') as f:
        return json.load(f)

def dump_coco(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w') as f:
        json.dump(obj, f)

def build_task_split(split_name, category_range):
    src_json = DATASET_ANN / f'{split_name}.json'
    coco = load_coco(src_json)

    categories = sorted(coco['categories'], key=lambda c: c['id'])
    selected_categories = categories[category_range[0]:category_range[1]]
    selected_cat_ids = {c['id'] for c in selected_categories}

    annotations = [a for a in coco['annotations'] if a['category_id'] in selected_cat_ids]
    valid_image_ids = {a['image_id'] for a in annotations}
    images = [img for img in coco['images'] if img['id'] in valid_image_ids]

    coco['categories'] = selected_categories
    coco['annotations'] = annotations
    coco['images'] = images
    return coco

for task_name in ACTIVE_TASKS:
    info = TASKS[task_name]
    task_dir = WORK_DATASETS / task_name
    ann_dir = task_dir / 'annotations'
    img_dir = task_dir / 'images'
    ann_dir.mkdir(parents=True, exist_ok=True)
    img_dir.mkdir(parents=True, exist_ok=True)

    for split_name in ['train', 'val', 'test']:
        coco = build_task_split(split_name, (info['start'], info['end']))
        dump_coco(coco, ann_dir / f'{split_name}.json')

    for link_name in ['train', 'test']:
        link_path = img_dir / link_name
        if link_path.exists() or link_path.is_symlink():
            continue
        os.symlink(IMAGE_ROOT, link_path)

    print(task_name, 'images', len(load_coco(ann_dir / 'train.json')['images']), 'anns', len(load_coco(ann_dir / 'train.json')['annotations']))

In [ ]:
def count_images(task_name):
    ann_path = WORK_DATASETS / task_name / 'annotations' / 'train.json'
    with open(ann_path, 'r') as f:
        return len(json.load(f)['images'])

def max_iter_for_one_epoch(task_name, batch_size):
    return math.ceil(count_images(task_name) / batch_size)

for task_name in ACTIVE_TASKS:
    print(task_name, 'max_iter =', max_iter_for_one_epoch(task_name, IMS_PER_BATCH))

In [ ]:
def run_cmd(cmd, cwd=REPO_DIR):
    print('\n>>>', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(cwd), check=True)

def train_task(task_name, config_file, weights, prev_cls, curr_cls, output_dir):
    max_iter = max_iter_for_one_epoch(task_name, IMS_PER_BATCH)
    cmd = [
        sys.executable, 'train_net.py',
        '--num-gpus', str(NUM_GPUS),
        '--task', task_name,
        '--config-file', config_file,
        'MODEL.WEIGHTS', weights,
        'MODEL.RandBox.NUM_CLASSES', '103',
        'TEST.PREV_INTRODUCED_CLS', str(prev_cls),
        'TEST.CUR_INTRODUCED_CLS', str(curr_cls),
        'SOLVER.IMS_PER_BATCH', str(IMS_PER_BATCH),
        'SOLVER.AMP.ENABLED', 'True',
        'DATALOADER.NUM_WORKERS', '2',
        'SOLVER.MAX_ITER', str(max_iter),
        'TEST.EVAL_PERIOD', str(max_iter + 1),
        'SOLVER.CHECKPOINT_PERIOD', str(max_iter),
        'OUTPUT_DIR', str(output_dir),
    ]
    run_cmd(cmd)
    return output_dir / 'model_final.pth'

def eval_task(task_name, config_file, weights, prev_cls, curr_cls, output_dir):
    cmd = [
        sys.executable, 'train_net.py',
        '--num-gpus', str(NUM_GPUS),
        '--task', task_name,
        '--config-file', config_file,
        '--eval-only',
        'MODEL.WEIGHTS', weights,
        'MODEL.RandBox.NUM_CLASSES', '103',
        'TEST.PREV_INTRODUCED_CLS', str(prev_cls),
        'TEST.CUR_INTRODUCED_CLS', str(curr_cls),
        'OUTPUT_DIR', str(output_dir),
    ]
    run_cmd(cmd)

In [ ]:
# Train selected continual-learning stages.
# If your checkpoint is on Kaggle input, update PRETRAIN_WEIGHTS above.
ckpt = PRETRAIN_WEIGHTS
if '://' not in str(ckpt):
    ckpt = Path(ckpt)
    if not ckpt.exists():
        raise FileNotFoundError(f'Pretrain checkpoint not found: {ckpt}')

train_specs = [
    (task_name, TASKS[task_name]['config'], TASKS[task_name]['prev'], TASKS[task_name]['curr'], OUTPUT_ROOT / task_name)
    for task_name in ACTIVE_TASKS
]

for task_name, config_file, prev_cls, curr_cls, out_dir in train_specs:
    out_dir.mkdir(parents=True, exist_ok=True)
    ckpt = train_task(task_name, config_file, str(ckpt), prev_cls, curr_cls, out_dir)
    print('Saved checkpoint:', ckpt)

In [ ]:
# Test checkpoints for the task(s) enabled in ACTIVE_TASKS.
eval_specs = [
    (task_name, TASKS[task_name]['config'], TASKS[task_name]['prev'], TASKS[task_name]['curr'], OUTPUT_ROOT / task_name)
    for task_name in ACTIVE_TASKS
]

for task_name, config_file, prev_cls, curr_cls, out_dir in eval_specs:
    final_ckpt = out_dir / 'model_final.pth'
    if not final_ckpt.exists():
        raise FileNotFoundError(f'Missing checkpoint for {task_name}: {final_ckpt}')
    eval_task(task_name, config_file, str(final_ckpt), prev_cls, curr_cls, out_dir)

## Metric cần theo dõi
 
 Repo đã được bổ sung OpenWorldCOCOEvaluator để tính các metric sau trên COCO-style dataset:
 
 - Known Recall -> known_recall50
 - Unknown Recall -> unknown_recall50
 - Known mAP -> known_mAP50
 - Unknown mAP -> unknown_mAP50
 - Current Known mAP -> current_known_mAP50
 - Previous Known mAP -> previous_known_mAP50
 - Known Precision -> known_precision50
 
 Các metric được tính ở IoU=0.50, theo nhóm class dựa trên TEST.PREV_INTRODUCED_CLS, TEST.CUR_INTRODUCED_CLS, và MODEL.RandBox.NUM_CLASSES. COCOEvaluator chuẩn vẫn chạy song song để giữ AP/AP50/AP75/APs/APm/APl.